# Reglas de detección YARA

**Módulo 05 · Sesión 39 · Máster en Cyber Threat Intelligence**

Este cuaderno es **todo lo que necesitas para la clase**. No hay que instalar nada,
ni descargar ficheros: los ejercicios ya están aquí dentro.

---

### Antes de empezar

1. **Archivo > Guardar una copia en Drive.** Así trabajas sobre tu propia copia y te la quedas.
2. Ejecuta las celdas **en orden**, de arriba abajo.
3. Para ejecutar una celda: pincha en ella y pulsa **Mayús + Enter**, o el botón de play.

No tienes que entender el código Python. Lo único que vas a tocar son las reglas YARA,
que están entre triples comillas y se leen igual que en las transparencias.

> **Si algo no funciona, escríbelo en el chat de la clase.** No te quedes atascado en silencio.


---
## Paso 1 · Preparar el entorno

Ejecuta esta celda y espera a que aparezca `LISTO`. Tarda unos 30 segundos.

**Pega en el chat el `LISTO` que te salga**, así el profesor sabe quién puede empezar.


In [ ]:
!pip install -q yara-python

import yara, os, base64, io, zipfile
print('LISTO - yara-python', yara.__version__)


---
## Paso 2 · Desempaquetar los ficheros de los ejercicios

Los ficheros van dentro de este cuaderno. Esta celda los deja en `/content/clase/`.

| Carpeta | Qué hay dentro |
|---|---|
| `ejercicio1/yaras/` | 3 ficheros que contienen reglas YARA |
| `ejercicio1/ejemploclase.txt` | un fichero de texto cualquiera |
| `ejercicio2/` | 6 ficheros: notas de rescate reales |
| `extra/` | una muestra de webshell y un `.php` legítimo |


In [ ]:
PAQUETE = (
    "UEsDBBQAAAAIAKOZKF1TzLp6DgQAACcIAAAbAAAAZWplcmNpY2lvMS95YXJhcy9hcHQzMi55YXJh1VTbbts4EH0PkH8YGAHiNFBE"
    "iroWCApFtpoCzdZts23fDF5GNhtZFCS6dRvk3wvKcrrdXvZ5AUEgZjjnzCEPR29a01mYtDg5Pjo+6rY1wvzD7TJf3LJguTKmVdzi"
    "Uje95XWNHdwfHwFs0PKnbgHQYYUdNhLhEiZra9v+qe9zYbb2ohIX0mz8Bj/3fkAC4tPAt/xONyuPS6tN4/EVd8jemss77HpPN57g"
    "zarmCvu1xxvlfdJoG77xJ3s2vrVr0zmqkksUxtyNCYW97HTrUF12hhal7WGQAd+777VCrzZcoYLP2q7hoPBC1fUI1fNNWw9y4ixh"
    "pApkIEQWxymnSZWqitOKRgFDghSzLBE8SysSJBKztIqk4DyNMxYmLGUjoFYOjKQVk0JxT0qReBFNuCcCEnlRlVCFQSwTFg0Fve10"
    "s+rH8z3pCVzCPcQpPHv2/YsSyNkPkbKEWQyMAYl+iD8ccKjr4jnaRWdkrlSHfT82eNIHA0d6BeEMymKsJCVcxRBRIIfIuIGMKVoA"
    "oY5yln+nYcPt6Eb9hTtb6hrfP7KELuXose9Z4PIu57LSNEq7yxtVT1u80P0yDoW20zMwHewDLBgCZ8AbNTqirsFUYNe4OT56+IWH"
    "Tb9bCi7vlDHdcrj6/6eJDxpg1GAaePX2w0+eTeI0oqTiGYqUJELwTDIlBZNJlSYiSSgGpOJMpTQIk4hlmCEPWZgFJEKWofjBs1wy"
    "yoRKPVHRyItEEngijqjHIkIH36qM/Mqz3HntHgiBWQBhCGkOJHSmcesUCuK+eQokAZLCjLhgOGwoSwhT9y+oW6QMygwoORQGjz7j"
    "e8uG1FVRMuAnbh0lkIbuEeQEyrmDcpj7bXTctkcuyIC8Z0ndOokc/p7B92HeyO5La1GB4paPxLjXVuRQlMDmUAbO/iGDeQwzCjCL"
    "IC5gFkLAgOWQz6GkcBXAg0OcoTRu9lgDqjMtVLrGltv1Uzj17ab1W95g3Z8eqJzGyU19d/1OvRGv7jqe9efP4+gmNy9XnIXEsvOv"
    "H/1t+nJRLdhary4n/yLB3TDe88ULaPgG4TTv5Jqjqc1K97YwCnWDj3Tu6d5DlEOcQZYCmTsl4RVEhYtAMgeaAQuBXUHB3DkWOVD2"
    "k7JTXUnTVHoFXv2IPbx9+laL8LpabN+8/tjuXrTzotntbhd6y8oPr8r8+mbnn2c383efspvzNW1W75Lb6P3fCybI135FXqs/yDtd"
    "1PwLdrnAjqttZ9f4opFm03a4xqbXonYyH2/2gDECVNtGDgfUH9qNXLvLX53W5LAlHrb8J+3vJtx0qxvLgik5g8tLILsKUVVcoht2"
    "+5TAn5LDKPxNZfWnyuofM3O67wAgcJNzeoJPHOwhOA7U6Ql/cjbEzo6PHr4BUEsDBBQAAAAIAKOZKF0SC7tuFQMAACMFAAAZAAAA"
    "ZWplcmNpY2lvMS95YXJhcy9jOTkueWFyYW2US2/jNhDHzwmQ7zB1dbCBjcOXSGrX9mGRFj00i7ZB0BZNYfAxXAmVJS9Fx8kG+90L"
    "yo9tix4EaWZ+M/zPDKi4axH2aIca23btqio/X429cH813cf1/eh5fn6G16vLiw0m8/bq8uLC4+Bis01N38ESJr+ihZGEawix30Bo"
    "WhzAVdV8W2/fwKn02TrVPzgOqX+45k+Yv28+w94MUGPEQ9S4xh8TKSHwNAel1Fgufz90HhO6ZGyL8C3VBu5679GDfckcXMNt3x3T"
    "pTwLuB5q4/v9cL3pT8X/T+wkN2t2qe5j7vP7to+N6eCXPtVjyJuEOcAIFTeE3jA9ugfXx+xXZLR2W4zrmAe+BJo9tRlqkvMkNRVz"
    "KDmXFbJgREUrDCQE5UpKpZqcaJppzwPXRnpXCqO450RUnPmKMWUUr9CdaZbpynFhvHU6eO0rop21XFDFheYkVIqeaZ7pwILRSnsk"
    "XHEWbCidll6WUkgVQknOtMi01YFJarjzgXFeaV8qbkwoSzRBclue6XKcjlJaeorEUiKkoUYFqRAFdVLYIL7qlpkmobQV49ozTa2T"
    "xlEuiLSWCeOYKe2ZVuMEtSOSVZ5oS6UUobTOofIqlEFRz9iZ1pmmpbLCOO6UNdxwiaWQqClWTFVo/qG7GpUI7Zwj1GrFrSC+VA5Z"
    "aYQrJRpFx+0MKTbdx2G8FcUwLhVd3cPjZGFXP3z32+3D3U9vFzd2teh6G1fjAcUwDrEJMC3Gm5LQtKmewWsxJJNgCfk1Lfy8CLN3"
    "XyYQdm2776M/JI8z3ddNizAtYr+HJWxehk/tOmBy9drEaF6mRcRh16Y3cPf7/c8/rj883M1m8AonbSmuFsmvHifzIh41yZOm6aGa"
    "i2gSrr2FaZHtDvfezmZgOg/Tb3CzTS//Cszg9Vj99j0ca45DP53psEsYVwu7usf4hPE6d7kb4MnEJl/f4TAnm5G4epy8+2/j41LO"
    "HeBzMhENuL4dlppA7PfDkpLcU5027bBF15jW1SYO0wI713v0s/njZHFzSh234frON/lHNq6QQR8g1bi5uvzyN1BLAwQUAAAACACj"
    "mShdoN+pc0oCAADeAwAAGgAAAGVqZXJjaWNpbzEveWFyYXMvd2Vicy55YXJhjZNfa9swFMWfW+h3uNUSlkCT2o4t211T1j8pDAIt"
    "acce5hGkKzk2USwjOQ2l7LsPqWlgb32S7tXhd48OktkqCTvJbSWVWmKeL5FSv9qKCb2zy40WrvYCeDs5PtrIjl2cHB8dCWnR1G1X"
    "6wamQH5JDk9eNYLS6A2UtZIWMM/HbdWeAVK63+T5aE8fbbQ4NP0IVxFHZ9uu0saB75U2NWtgobvKHwnWSXcQBWF8HoTnUebbFrVx"
    "/TRwVcVsFTgVDVkeoaSTCc1lVLI4D3NZBmWZYhKGNCUf6tCpgzLheTTJRJSFHCnDcBIHlPMoZhixhB/UkWdnGNAoF0HGQ0rjMuGI"
    "MhVpmZRpKKLooJ54dpwhYhDyLJ3wOBBJijJKWIwJlSwNvRPbmbpZWR9wz/ohAHUJg22j6mY9WN7/mM+WwyG8fdd8iUqyZjD8BhIr"
    "DQV5rliztlBqA1tbN6tDrPAyLsi4Z6sXb6lnvSGAgnjFkjNco26acasKMr1ixrDXQUF+esjjbDEvyFlBWmkU9FvWVdCvtO2g32rT"
    "FWR4RqDcKrXTRrzjY4e/5Obq8vn6Zj4D270qOS3IzcPibrYY3T7M59ePT7MLQK0Ua60sCKBU6qllWDeraQBcGyHNrVba3DGznn6h"
    "dG89fbcOUlnpkjntCdYxmLrLrmRn9dagHPR43YjfBbEGC/LHBbbP6JY1XzsQetcozQS8y+2enf0fiwO3dSs/H4vSyJRLBfpGbnQn"
    "K/3Bzj/pGz/v+tShUTeidr/QP5oIdAldJTcnx3//AVBLAwQUAAAACACjmShdqAXoZAUAAAADAAAAGwAAAGVqZXJjaWNpbzEvZWpl"
    "bXBsb2NsYXNlLnR4dCsuLgYAUEsDBBQAAAAIAKOZKF2ykuDC3gYAAAAQAAAWAAAAZWplcmNpY2lvMi9sb2NrYml0LnR4dK1XTW/k"
    "uBG9G/B/qNtcnPakpZaSAZJFJgMERjZYIDEwyJESSxIlikWRRX0d/NsXZHevPVjbB2P71I0WX33wVb2np6cn+Jnq4atiyA6fgTuE"
    "hZyWnzw0wjN6BmEkjOQZPItKIzhhPI2LcAiNoxGOn//816enp9ub25u/xw/8n4IDKViA8uCZNJoEgqZ2m2WUh9ubhwY2CiDJfGKw"
    "YkuRz8h36Xs6vyitoUKwodLKdyiBDET0x1/+C1K4wSCDV4z+AP9GtKAMjMpI4E4wkKkxRrkkI6xF4fwVQqMY0tk7UAw1BS1jqIpC"
    "2zFU2/lkTaNFVkzOg4i92MBjTUbegadL+h16xYIRGnIgQJNpgdWIB3jsEDyRQZeqfa3O57/PwWKEa9leNHiIfX0kB18dLR4d/KzM"
    "4L/c3nTM9sv9vaZ6qBQLy0dZZoPTFS7tfCx5CrpvV5dla7Us3hZu2BQylSE3NQp5IKPIvAazNRWXuu5W7EObl8M4zdO0znNvp2HE"
    "ec51JnadtVtht7dhsnyYnbLFSv2mqetWt/jZ7rJpWn/ac1tVflvM7ivZhultmNOa70Nf1dO4F42THdbT1AqJ86aWaQ3D4L01Wslt"
    "luU7MMW8nkrOEKeemqWtWz0GdpnITmZrZxp6cQqhrpXNt+GdosqjWk4n07dmstvYtn5oT5stT24rnXIs23wsVX4Ujn31TjZi6XUR"
    "ZGdldswCdjioTXBfNFyv47DgyePu82ZqbW/V2zCVVKKf2OraqXZvZW/dEtphCFxkppq3ozy5fDmKdsPhnWzqo5pywbjsR+WxOE5F"
    "tjQDb07nEy9hOE1iPRbHoeW9f4a5vUlETJw35EahoToz9I/i5kFvfxA9X0f6CENfR/oISV9H+ghPX0f6CFVfR/oIW19H+ghhX0f6"
    "CGcT0lWjvkeBaINwwjBiFCru0OFZOJYognGz1x0Kjrv5J7i9+Y4QNS8ubdIyKuMLJRRNo7SKImAdtU6MUWbio1YLg3wHhrhTpo2R"
    "RnIIarTkWBiOIc+K5NAGFhxThUswQwwCLGnFqhZabzASq1kwSmgdBZuENeYbka4xUoAEO5LB7QAXtbViu0sPR32xjmYlk0DCorgD"
    "iUmeFRnw1PC5LCMhlurorFwXNY96eoB/NPyGsJ0xtYYpqHqIWYsBAWeMGTm8pvXoYnu5izZBcUilg1ej1RsIHwsXSgI7oUwsK66a"
    "pJN+84wjCDkqozw7EeX5DiqsRfAY5Vx5kAGB6SqszpKLt2OQF3JDamyFEdU6suj0BjWZRrXB4cU+xEZFSiTTwwSCWdRDBDzAL8GB"
    "RZP8kUc3qxo9+O5qIlLeffAMWg34Q4u80MIp9EDNO8Uc4F/IQDO6WE28hXg+NkBxus4FL+ajVfM5gLheIDkgBxI18g/25+VtRT4C"
    "mSsVxAbBR/MUM2wCB4eHaOOgFgZa5AtlTRMXfbolUVHgeIgMPGgy8J/gh08eHhfFMU6cWP/l/p7Pvw81jfed8B2L9jrGPzV/02rG"
    "2xt4YRzBYOw/xdtgUacQZxKm2lLOjdJnr9U4xJjA79zgmdCpdovOkxEaHr7F6f9Gi9EkZAJVxrPQGl7aq2viy7IcmJx11GPNB3Lt"
    "/e3Nd6c4cSH2qY4kcUTjeQaF4rMBjGbXL+jiqH3S+sWWEeDQWzL+4pwTlS6zmeoWEIyaAsLDt4RVkzufkBi9bKoqxFUl+IcLHAwt"
    "5yu5A0at0wiY37J8HvoWDaY5ENHGOowrTpzzThwy8plQcSwfvsGsBFiXdk4M+KfobGHEkfyV+b+RKbKlifY7ciON9cM3qELaoAf4"
    "H40YD/vn9dCJObUzNc/TiMk3p3QuC1Fv0SYrHyf6Ot8Lng9Gq81xjtJEXxbhZUq7YKRDmcbsbKvjzAlHIb0dXN5yXjfX59Z3gn/n"
    "ZHywosSsyu1g8rEdWuqdPqmpXY95rau9ztexVEWPyF6J7C3T5YOVC5myMPs27JWurVbrwj43+U5Yh3Y/VquoWNhpHvfpHRhz7IoK"
    "j7WZ7Gzqretz145mMXleZFln9n3kVc49TdqWb/laHyyV83yaa531q/VyVmSXWXjdT7XnbSyw6bqCdizrYu3fenXwwU5ZWxylUccm"
    "K7xxssrNadqnmeaKBz6tTbNk0ol1KNrlnaKmZhO1kq4IlgvTdZuyoZ+FqHSoAs5rX6xb1jjuZpe9UxSXxmWNKLBct6rM9FBUS+Fq"
    "g1NHW6WNUqLq87DMu7DvZBM6v3S52mcKNa2+MsTD2E6FDK4th0bVbRGybG/mKaN3slnr3rDqqlFw7lxXDkyLsv447dvSFTu6k8tW"
    "0XSz2rqXLjttxg9/fnglf7EKv8DX7PM//1L8ClBLAwQUAAAACACjmShdti31DeQDAAAACAAAIQAAAGVqZXJjaWNpbzIvcmVhZG1l"
    "X2Rpc2tzdGF0aW9uLnR4dHVVTW/cRgy9G/B/YE+92PJn4CSX1I2B1pcgqAMYPRVcDbWa7GioDDkrq7++4Izk3bSpjhry8Q35Hud3"
    "CoGb05PTky+9F/ACD152T4rqOcITtTl5nUvAc48KPY4jRXIf7M85/Mk5QSSdOO1gQoHICmJZ1Lyef6rn5/eq2Pbk4Ek54ZZKQsvD"
    "mHjwQu5QxTEJqBEaCOMHeO4pEWAiGGbofCBZ6t+HALPVcKgIPQpsiCJQbNM8KjnA6KD3zlEEjoAgI7UeA+w55KFyfMXww8hJMVr9"
    "Ng8UVaDHPVVMx1MMjO6YZ4sRHsExKEOilveUjKGRWQg+doYNk6EeBb1yPivHpYoyCEUHl83lW/jVa8s+2s/ShwlDIAV0LpHIewPf"
    "tFff3t2q4JtO9tf7IcpNvvs6vGD39024C+++zf3X2+nmZffOou/DhLOA47wJBG1P7Q60pxURpp4itDzOPm4vRhT1cQtef7LvddaV"
    "aI7qQ0m+eqs9cAf3Y/IBri+vb19vYccjztbF5vTkvlNK9SYOler9yyBh8iHAhgDDwKJlCCLeSB76dei49JyDq03HAvr4n3rH2nx8"
    "AC/v4frqprm6edNc31w2V3dXdvw5EAoBDegDZKmUHh+KYhYgaDl2Pg3VDMql7c7LTswfl/EXy71WTk3LwyLIwmmiNbXCrnhFCnbf"
    "RC35PYEjRR/IgY+iKbdWSEypPU92f0dFyIDHOm/gcbGIuS3Rt+zNHXEGpbaPvsUAsvMhSLmMV7O140jQodg0Tk8+FuGKGpcsxVYR"
    "V1M9cAF2FEgrbp2Use98dMavkPl0/9QcJWiaf6hxyGJiMiDhTifzMYrxWnohOSj4CCOlAaM1qqQFFjnGH9j5bi441b4CnECWZTIy"
    "B/k/alPyShAwbQlw4GzW5q6WUV6oetktQpur0Ndl8+8FsPTpmYw7JfNuT5D8ti8mD4S7woxCOEzuB9vlDHxdD47jz0Vtiq1ClqaK"
    "97eHz39Aom0OqJzqND9mUR4oLcbBIGzuiay+87bwNpy18NkkwrZvls1eov0whhl6wr3dLJIs97XtWnfZJluDXfXUpi6hg59KEKH4"
    "MP8wErrEQ3FJrzrK+4uLaZqagTmOOJtJLjZ5vthoewgY8aXLYT07X4AO54FbDMtfWcP+Wn98X2jjI8aWShTFgrfsUaO08tfex51N"
    "X60vKIDQVROWh8vrDJidr0Z5rs/OmLgjEc8RgzSwvpUI5ir1gzkZQwMlYVW1clo2XdGZHwZyHpXCvKyu73fkmkl7irWrJg314/FC"
    "EE0Ut2rLuiCvb+9K/cyCxmQYWq2bNZvf7O3d1XmbIjDuDKD5B1BLAwQUAAAACACjmShdx5RNrqIAAADBAAAAEwAAAGVqZXJjaWNp"
    "bzIvc2V4aS50eHQVzUtOxDAMBmCO8q/Ylb6StGEkJFazQCw5QGq7D6kTo7HRiNsjvgt8V4Urdvdve23bTdzE7ND6ovetvYD1UU8t"
    "jGcc1byc5wW+S0VhRhcp8rKM0q2J+mlkDlTClOM88SJE1I8p5jyva+7CEOeeeMghppRCFJpyP/zfv/pzB2n1Qm4olWFSGQU3MSub"
    "4HH4Dt8PAylLLTdB0zRveP+4fn0+/QFQSwMEFAAAAAgAo5koXfmuo8oJBQAAQQgAABMAAABlamVyY2ljaW8yL25vdGUudHh0bVVL"
    "c9s2EL7vr9hbkyaU9X6dSlGUrdiyVFFO0146ELkUYYGADIAy6V/fASnFzkyPILjY77XAinExRcpPmowZ/3HSyiqZMy5ascohs/Zk"
    "pjc3ObOaly2rbsDDWaElKo25KqRFmxG2uFHgYaBOFcaaxUdMtcoxXG22YRShVXhgOSVcg4d36/X84WmFmgQxQ5goMiiVRcuOhDHT"
    "BCetYjKGTAuXFpkwChNu2F6QQV+XTP5mwMNV9fMMbnDPDCWoZI1H0isZC51xmg3Hfjm09tYfhq+H8mAyVoyDaj4vbierHy/ADMZK"
    "Ws33hVUa+bW+tNfDIVaFSBoUKeMCmUwwZoUhR9VkZFqQFjYjjfOto2WI6Tj7ioKfuTxgrIw1UMPGfYWFPLH46DZcIyopLqxjBjET"
    "Ys/io6kbpIWMLVfSoM2YRRMzCWfShiuJKq1rnaSNzt0erlxPJ4oT0RSanOiOGYst5oQ8xUoVkCjJLKFRboWxg+RoUMItJcicrIEm"
    "9wuL44u9WhWHrO7ITidQGuOM4mPzobDKi53r6uTQIpfg0DduOZJMCHSZopoN7h4i+B0XdXg0IZep0jlzW1/xWXGJeQXBux8GDZcx"
    "4XKxvdrRAg/3opA2Z7LfbeOnhlKCXbQ8J/MZLvLEhbEqb8Aa5NLwhNwGuE1TGUs5XoJWyytJ11qDh3P1KoViSX1OKCgnaR15PHMG"
    "75a9W4N0Jl1hIgQEtZjGyb/S/94qkexJY6o0ZtzU4tQ8r7LmFcYqzwvJbQVL+z/T0CRPpfAh77HKT4IsiQoTSgtDzjhfCBebxOAr"
    "F8IZW7iRSJWGqDiR/hj0D8LWmMpxYwiTSla5KgyeSJ0EwS7jxo3Xa1ahUTldvmPGzvWc7gXlrqGtE8L1R5u2hURDtji1qHRHJ84F"
    "y4QA/+EBN8utvwsj9IPtOopwdxdiFPrwM9k55UpXdVnjFsz5gVsmWv4SP7nckG5ug8/wMaTLOQouj7gnoV7hC/4Q/Ez4BSNLLMcv"
    "GNS5qG8rLUlwg2cmccNlaUmThPekNDguGWnBQuk3hndK8zclsY+b7XoTNvLti+o60a7GtGCnrnl+93cKXvS0CbcYrB932+Xsabfe"
    "Rh54WHY/CK+kq/eusQuu1R7om8s1TWclCjc0wKUlLZnAjMlEkDat65VNTX2LK9CUJNz+ggPvlVFn9Y0ivefPrDZkHj4+fV9jFDyt"
    "oKxFo7wQzCrdAu9XyI7oxUvSLZhxGzuyMTPZFJbNBj4qSwa2l8g2K2/z8y7wwLt1+i5lqpwITOxJW5WPwZuvH/3dcv3oNX39YOdB"
    "f/D08no/nMxeosmfi+rwXb/98+Pbt1PyNqBNuXp5Oxhol/t9L+0vBr1Rt5Ow4WiYjtrhYNIbj4LxhNiMEUw6d7e38V+PRZLoxdv2"
    "/vZA4eZ5tYien/8+fmOZfoaXl15W6XMymgy7w+GxSkflUWTtUhzbZTHoCXmAl50qpxguZmE47oe9eW/R7/bH7f7AD4LOfDbsjGHY"
    "aXd6/U5/MpkPR53efLzo+v3ubDIKRgNIFjPZf36b/HOfvPbO8/NMhYP72/vN2wP0/HbQ6fcWi3FvNpkMJ/2B34GbLx7lJ296eYfB"
    "88If9dsaRp4HnjcPP66Wzbtbr5yzd/4uhEORvjxncQo/J3T5GD097CJYKUlaTaE/Gcej2ax+4fd7Bk3AXVr8Xt6B2S6YQri7m8Lj"
    "Yt2CbrvbAaUP/wFQSwMEFAAAAAgAo5koXenvR2o/BwAARg0AABoAAABlamVyY2ljaW8yL25vdGFfaHRtbF8xLnR4dH1Wa2/iShL9"
    "7l9xxJfdnR3AL2yDGGv9vIl2BiLCKIp0vzR2g60x3R67ScJE+99X1Savuas1kUg3VadOnaqu9pKh6vj+y6hWFW9a0/5Xz4tTV6vz"
    "8TwR7MhBzyj8vz9jOWUhlrsuNJZ/wTvLU1/xRvGOl5NCHt+D/eW3D1Bt+AlbiUIKxQqFU/8ZRceZ4mAQ/BH7jnPwI6sbsKKQJ6Eg"
    "BVTF0deKL4BXMpVSbb+YTttOKinIg6KNwo/rS9gdxQ+x7Fsm0Ktzw7+M9lKocV//4gvYdvs0Cn+WfluwvShk/3P/8+RUzNsXT27l"
    "9Z2vfj2yw6+2KAtxVvWjs/MeWMf4z4fZuZxIUUuxnBJ4aNxxHJiqKH1U9aFqzpTtvi65UDVrpi3veilYg5IpNsG24j3X/4N1HMWp"
    "67hQzRm9kgQhhc6Aoe3qB9Kp590D78ix7i8LPNZNgx1HfTzysmaKN2eUvFedPPMSbK94h7M8dWjZ+ciFmmjMW0mbKJgAf2p5oQYb"
    "zUVJwmtPu6YumjPYA6sbtms46qEagrMO+5M6dXyi0TTiSqKXe/VImby5SIFaKN4JrnQwahQKNcEd10kToBTNGVLwHtpHSQ14L09D"
    "cuTXc1Hi1MMeOxBSjOtjKzvFhMK+bshRlHjkg33Ji+7cKtQKe9npttKA13udc8mLuqQoEFKRKp9fPTvecNbzj1oMQkB26Pi4503D"
    "u0HDuwvznvMfOErBz5oG+R4ka1D3OoKSKNmRHS6wHW9PiqlaCoJsO/7AhRr45bhff0e6Xv1ti2S92kbJFt9vcXe9vbpewbdxtf6+"
    "uf2Mm811kuHu+utXxBmurv+4yjaT5XQ3FGK5C5PXI6YVaLu64JrbgasXeYjAS8FenTVAtLpHtN1m32622K6xyW63601G3DbIr79m"
    "AyNsr6436fgm2mzvcbvOt3fRJnt/0LtxUTGdPmse2bnXTSqg5Kmo3g7zy9cuNEjrTj5wKofuoqEZcKgfLtoNtd6x4sdb37kBLBd2"
    "Rn9zE0kCK4MVwYsR+XAiZDHiGGZOf8HMMCO4MaIE8xieibmLLIU3Rxxo4xxRjMRHGtFOmpCcUdMM8X/vuoo9cOw4F+BCi8pL489B"
    "qGT97YaEXGXbu/Xm37iKbhFn2Qo32SrbbqJtlmJqLMv6AXX5ZaTYbkyDkQtljVA0rO+/jC4bo9BI11itt9hkq+hbhmyVbO5vCEGX"
    "400LTXJ36mvB+x77Th7RnYSoxeHN5gL1bZ1e5/f/E8pYU1cfZVnva15O8PfNbfTPKLv9xyuGbr2bbPMtWmWr7dd7JOvN5vvNFtfb"
    "AaGXzUvN2k7uGn58I6D4kxqrjol+L7vjAqe25V3Bem4s9XSGOrf8y4jMpkXfj0LjUXbleNdx9mMB/TVmTTPglLyQnT5NC5oL3Jgo"
    "tusxuUiHCVk9G9NP2po19UEsUHAaSYP4F6mHET26sLvYvRjq2+KR14dKUZTuyBqaTi/9SL3asz3/4Htx/Y0Ow/OApjNdoFasqQuj"
    "lEKhqJg41OIAVdW9QU1+6ORJlONCNrJbGE0t+Li6sLDN9unjjjNrn4ZwFiZDOnj+Pf7zx1xc0/wIYk1s493d6FCUd2vr9zXFfH+X"
    "Ou2TUdZ927DzArtGFj+MI+sOtViAnZR8l9XC2DeSqQWUbI334wdvg+z9UPrzdXs8psMyDo3pp/F4/GlqjMeg5XIXTo3/GEtjFOLD"
    "4+cIPAQp7Ai2A9eEY8GdI3fh+7ACeBGCCJGLmY/Y/NAXFWflCOFyF+qCv17g1+lCE5qW9UNozFKYDmwPiQczgJ8itZDZyDKaQbYL"
    "x4brIZsjpnFjeBG8BEkM34SdIHWQBYgCWDnNHSclmokNz4UZI8kNa47MQxzBSohvZCH2MAsQJIh9AjEzuD7SHI4D26GZ6Oa0mThE"
    "bO4StzSjdLOcXLIUfgY/IBfLN/KMQFwfnqcpaeOZR7TTGaUzC5BHyHPMZ4hMw0yRO/AjZC7mWtHApYhxhiBGNENuE8m5h7kFLzA8"
    "F3kC10ISwJzDnCGNKSN/TtrPdaZWTPXJcviB4VrIHKJnZpin8HzMMuQBpW8HsByYFqKc8OOUKpbklGOUkqqmSyE8j6S2bZgJ4Vi6"
    "FoFDZU98I4/pDnBikjQ2MfepFyIbSQp/RiQjl+4A0moOKzLSAGkKJyCeSYDEIg65i9xDkOmfHK1eQiFM33ASeA7mc1raJtXXd6gW"
    "MxupjXyG3EKQw/LgzpDlRu4Sn3ROTJyYBM9jyi5L4cxJfDuGl9GVZs/0nfT+XfaxLlW1sEyTDuf73tWzYBQaLSvLWhwWsGft03Bm"
    "fzcbhR+3+BPdOr9Z0c4u1C/oQ/NfTiN/UsPLEp3ClwlEB+f5LbSO+oHI20rzHt6gL6/sU51aaFTWZ1TOZYpZzzQRZHkmg0odm9BY"
    "Dl+0SaNtRh8jsHZ87xuXqZnmpmVaRprTxzD1Y5R703JMYzmg7fVjXM7ysurC/wJQSwMEFAAAAAgAo5koXfAO1Y+7BQAAtQsAABoA"
    "AABlamVyY2ljaW8yL25vdGFfaHRtbF8yLnR4dHWW32/jOA7H3/VX8M13QGInk3ZnNnCN67Tdmy66neKaw9w+HRSJjoWRJUM/4viK"
    "+98XlJM27s7mJbBMkV+SH0oupdqD0Nz7q2xrtcyq24gQLHDwKKJTYYCtQy6aGXCtoVYaPVgDg40OhG27GNBBw/cIW0QDaIQbuoBy"
    "BrV1IDE9Kmtm4NFI4Aaw5UpTjOjXUPqOm5OClrvvWbXl4nvs/tFzVcdc6LgtCzKqykKqfcVuv95sfn+6gy+b3x7g6d+fH+5vIJsX"
    "xbfVTVHcbm7HFxf5YlkUd48ZZE0I3boo+r7P+1Vu3a7Y/KtoQqsvCh+cEiGXQWYVe75//OfD3f3j8+b68ebuKhvQZ+x58L+hiVeZ"
    "sVnFbqwJXASIHriR0CP0SmvonN0riVQU6FVoQBkfXBSUt0912MZBmR18VkFYZXJ40sg9grEBITQ8wHYAMTonu9AoJ6HjLij0MwgN"
    "grA+QMsHUEa4tFkeOyWlokBcQ43oc/h2VNWg7pKkYKGLTjS0aTsqSDJtDBCNQYHeczeAVHWtRNRjUGowHjp0Co1ACb5DobhWPvjR"
    "f0Ctk39lQGKgpvIt+SS5nbPkNmcbe4JgZCYRNDuWSmswiDIp5ANwEOgCVwZ4a6MJ5Pko2OewafCMJ3A80HOHRiYiKarvyJutT3iO"
    "3UotiT6H65pg7fjQoglnGhwKVHsk6MckIVirJwCb3V+wn7My6uoX65KAgD7MiAvBzWve1iD49jQ+wD1Vx9ak8y2f1DdpCYnXjart"
    "rAvchGNsGV2iAz0JmoGPoiF3/3l4noHkgW+5xxFNGxp07x2w8k/j/hnBR5dASrnXA8Hn4f6Wik8pNcglOhJLTxoD1bBv0JwD+xej"
    "fP27735ZPMrpBJdaVXO4HXMNbqDgE0YoFYh+OgkDeFuHnjukgVA+DYNGnuhRzuEenVdbjeN2bb3Py0Krin1DsEYP4BvbjwMxArrV"
    "2BLLoRnDvp54VMAdBnDYcydRJhLIouFOQm/d90klgwoas+raA4dd5I6bgJgoaPl3oqp2eI7upAyPNvGB2h/HdovAKYtgwWGI7sjb"
    "SAAeBHZ0/rBeGWn73KFX/8ON/ZsXDtHkvZKhgTksF4sZHNcaVLsmjIt/T8ormqUpzxL3SiCUW1f96Dj/UcJfbJ84v6fj7XS2+R9Z"
    "bqhfysMe3fAG5RQEh4a3+BZx1PfWQkMNBtEgD6lBe64jpq447GLgVFlWpoApvWNCyr+5LIvxNbmT1mSBEPYq4R2dR71HD8KpVhmu"
    "PStbDBxEw53HcJWNFffz5YfLZVax2pqwhuVld4ANb2zLZ+C58XOPTtWs9MKpLoDmZhf5Dq+yX5/TSlZNCjRO13+3NgTbZhULeAjz"
    "4LjxtXXtGmLXoRPcI9taJ9HNR8s1LBfdAbzVatqadKNwjS4F8mEgkoYOrzLyXAjv3wlIG5Sp6Xo7hgi2m/g/otbaPYF2uZjB5ZGj"
    "aRZZxVrudspMRL5fu6S1jm4ts3sz7A7s/ubr41XWeoUHFDkeMBurwbXamTUINAHdO28fydkoOzkByX2DkmllcD5yv4YPl90htWve"
    "H1fo7JvaJFWTldWfVpar7jCR5GidlV821+vrp6eH+5vrzf3XR5ZPmgovjC7NeerFGow1OIqhwV3D8tNJ3Ph8cVYyakQSdm7/8d0z"
    "ieqsT98A61odUJ5vhwU7yecxWEYfVztno5HrUxPIpiyirljJ/j/mp4zERPd5r16xOPnXWIcfrlC9X51/6A5w+WpynJgFyxN4eRpI"
    "eDk90rUBL2+25HvSkDzZdS7t6Zza01fAaXvU8MLKYhy96tSHM38LltfW0gX2wvI0JfDCjihREVKPKpYTH2RylJPTfLxGeWGp72kD"
    "RahYWWytHOifPisrVp7+0lt6By/sbvnp08cFE1Zbt2ar9GM/rRaLxYLV6cc+X6x+/mnF8OOFWAmGol7US1aefI+XRtTVH1BLAwQU"
    "AAAACACjmShd/IV4emMBAADdAQAAFQAAAGV4dHJhL211ZXN0cmFfYzk5LnBocDWOQU/bQBSE7/kVI8sHRwKH3kpxjAy4Aimx2wSr"
    "7Sl63n3BW2285u0mAaH+98pATnP55pvJrodumMxmWDbl+nFV4K7EoripV8VjvXqocY6qRrlGUxX4Vd6s78vFAt+b6vahropFOjZv"
    "XR8M9wzvrIMlD0Wa+/cUUoHF+GAUeWi2GKhnC3V5eYaBhPC8Zwx71uRH1yCuJYElCD9Zgjc9dtTzXxIop82Tw46sUcZ5B804sGjS"
    "6YRV5xBlbX5f/r5rlj++ZbM2z3rXSh5dTcwWSbw1ln1gsqGb4i32gQLmGCOJdRpvp1f/ThrFfWDJszZfsxxYzkdq73EgMdRa9h/6"
    "dkTeB46dsYwkFnfEHLtX/2w3Ww6q25AIvSaxsN/bcIbln/XPxaZqltMp3vA5FyTPgs6jNJbTg8AvgYQJylk//3oBcUc//3KRR2kX"
    "dtYPrAxZ1ZH4JOZeOc16mkbZ7FTMJ9f55D9QSwMEFAAAAAgAo5koXeAEsTOkAAAA0AAAABIAAABleHRyYS9pbm9jZW50ZS5waHAV"
    "x0GOwjAMBdB9TvFVZQFSRcSWliKxmHukrZkYJXGJDRW3H83bvfG2pc2FgB9eEjVBpl82LnKBvbFItSYZK+ERs4piE2Xjj+jJ+UVq"
    "xRWVdpSvvjIffBK1Hv6t1Hr4Lar28Ot8HBwtSdCN6TzdmeqHKq+CiMLYaR5DOk/d4PSrRuXQrdGoOw4AQoA85QJSE1gsM1P9TwRV"
    "qDxsj41QpZWY3W1yf1BLAQIUABQAAAAIAKOZKF1TzLp6DgQAACcIAAAbAAAAAAAAAAAAAACAAQAAAABlamVyY2ljaW8xL3lhcmFz"
    "L2FwdDMyLnlhcmFQSwECFAAUAAAACACjmShdEgu7bhUDAAAjBQAAGQAAAAAAAAAAAAAAgAFHBAAAZWplcmNpY2lvMS95YXJhcy9j"
    "OTkueWFyYVBLAQIUABQAAAAIAKOZKF2g36lzSgIAAN4DAAAaAAAAAAAAAAAAAACAAZMHAABlamVyY2ljaW8xL3lhcmFzL3dlYnMu"
    "eWFyYVBLAQIUABQAAAAIAKOZKF2oBehkBQAAAAMAAAAbAAAAAAAAAAAAAACAARUKAABlamVyY2ljaW8xL2VqZW1wbG9jbGFzZS50"
    "eHRQSwECFAAUAAAACACjmShdspLgwt4GAAAAEAAAFgAAAAAAAAAAAAAAgAFTCgAAZWplcmNpY2lvMi9sb2NrYml0LnR4dFBLAQIU"
    "ABQAAAAIAKOZKF22LfUN5AMAAAAIAAAhAAAAAAAAAAAAAACAAWURAABlamVyY2ljaW8yL3JlYWRtZV9kaXNrc3RhdGlvbi50eHRQ"
    "SwECFAAUAAAACACjmShdx5RNrqIAAADBAAAAEwAAAAAAAAAAAAAAgAGIFQAAZWplcmNpY2lvMi9zZXhpLnR4dFBLAQIUABQAAAAI"
    "AKOZKF35rqPKCQUAAEEIAAATAAAAAAAAAAAAAACAAVsWAABlamVyY2ljaW8yL25vdGUudHh0UEsBAhQAFAAAAAgAo5koXenvR2o/"
    "BwAARg0AABoAAAAAAAAAAAAAAIABlRsAAGVqZXJjaWNpbzIvbm90YV9odG1sXzEudHh0UEsBAhQAFAAAAAgAo5koXfAO1Y+7BQAA"
    "tQsAABoAAAAAAAAAAAAAAIABDCMAAGVqZXJjaWNpbzIvbm90YV9odG1sXzIudHh0UEsBAhQAFAAAAAgAo5koXfyFeHpjAQAA3QEA"
    "ABUAAAAAAAAAAAAAAIAB/ygAAGV4dHJhL211ZXN0cmFfYzk5LnBocFBLAQIUABQAAAAIAKOZKF3gBLEzpAAAANAAAAASAAAAAAAA"
    "AAAAAACAAZUqAABleHRyYS9pbm9jZW50ZS5waHBQSwUGAAAAAAwADABJAwAAaSsAAAAA"
)

BASE = '/content/clase'
os.makedirs(BASE, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAQUETE))).extractall(BASE)

for carpeta, _, ficheros in sorted(os.walk(BASE)):
    rel = os.path.relpath(carpeta, BASE)
    for f in sorted(ficheros):
        ruta = os.path.join(carpeta, f)
        nombre = f if rel == '.' else f'{rel}/{f}'
        print(f'  {nombre:32s} {os.path.getsize(ruta):>7,} bytes')


---
## Paso 3 · La función que vas a usar todo el rato

`escanear(regla, carpeta)` hace exactamente lo mismo que este comando de las transparencias:

```
yara64.exe -s MI_REGLA.yara CARPETA
```

Y `escanear(regla, carpeta, recursivo=True)` es el mismo comando con `-r`.


In [ ]:
def escanear(regla, carpeta, recursivo=False):
    """Compila la regla y la lanza contra los ficheros de la carpeta."""
    ruta = os.path.join(BASE, carpeta)
    try:
        r = yara.compile(source=regla)
    except yara.SyntaxError as e:
        print('ERROR DE SINTAXIS EN TU REGLA:', e)
        return

    objetivos = []
    if recursivo:
        for c, _, fs in sorted(os.walk(ruta)):
            objetivos += [os.path.join(c, f) for f in sorted(fs)]
    else:
        objetivos = [os.path.join(ruta, f) for f in sorted(os.listdir(ruta))
                     if os.path.isfile(os.path.join(ruta, f))]

    hits = 0
    for f in objetivos:
        nombre = os.path.relpath(f, ruta)
        m = r.match(data=open(f, 'rb').read())
        if m:
            hits += 1
            strings = sorted({s.identifier for mm in m for s in mm.strings})
            print(f'  DETECTADO   {nombre:34s} {", ".join(strings)}')
        else:
            print(f'              {nombre}')
    print(f'\n  >>> {hits} de {len(objetivos)} ficheros detectados')

print('funcion escanear() lista')


---
# Ejercicio 1 · Una regla que detecta otras reglas YARA

En `ejercicio1/yaras/` hay 3 ficheros. Ábrelos primero con el icono de la carpeta, a la izquierda:
son texto plano y se leen perfectamente.

### Nivel 1
Ejecuta la celda tal cual. **¿Cuántos ficheros detecta? ¿Qué string ha hecho match en cada uno?**

### Nivel 2
Cambia `escanear(regla, 'ejercicio1/yaras')` por `escanear(regla, '.', recursivo=True)`
para que entre también `ejemploclase.txt`.
**¿Lo marca?** Si no lo marca, ¿por qué no? Fíjate en `all of them`.

### Nivel 3
Añade `and #s1 >= 2` al final de la `condition` y vuelve a ejecutar.
`#s1` cuenta cuántas veces aparece `$s1` en el fichero.
**Cuenta otra vez cuántos detecta. Te va a sorprender.**


In [ ]:
regla = '''
rule YARA_rules
{
    meta:
        author      = "tu nombre"
        description = "Detecta ficheros que contienen reglas YARA"
    strings:
        $s1 = "rule "
        $s2 = "strings:"
        $s3 = "condition:"
    condition:
        filesize < 100KB and all of them
}
'''

escanear(regla, 'ejercicio1/yaras')


---
# Ejercicio 2 · Detectar notas de rescate

En `ejercicio2/` hay **6 ficheros**. Son notas de rescate reales, de familias distintas.
Ábrelas y léelas antes de nada.

### Nivel 1
Ejecuta la celda. La regla detecta **4 de 6**.
**¿Cuáles se le escapan?** Y ojo: **uno de los 6 no es una nota de rescate.**

### Nivel 2
Añade las frases que dijiste en el ejercicio 0, como `$s20`, `$s21`...
¿Detecta más? ¿Y qué le has hecho al riesgo de falso positivo?

### Nivel 3
Cambia `2 of them` por `any of them` y vuelve a ejecutar. ¿Qué pasa con el señuelo?
Prueba luego con `all of them`. Quédate con el umbral que detecte las notas y **no** el señuelo.


In [ ]:
regla = '''
rule mal_ransomware_note_generic
{
    meta:
        author      = "tu nombre"
        description = "Notas de rescate de varias familias"
    strings:
        $s0  = "Use TOR Browser:" fullword
        $s1  = "your files were encrypted" fullword
        $s2  = "Where are my files?" fullword
        $s3  = "recover your data" fullword
        $s4  = "DiskStation Security" fullword
        $s5  = "Your data is stolen and encrypted" fullword
        $s6  = "ON RECOVERING YOUR FILES!"
        $s7  = "CONTACT US WITHIN 72 HOURS" fullword
        $s8  = "YOUR COMPANY NETWORK HAS BEEN PENETRATED"
        $s9  = "The decryption rate depends on the speed"
        $s10 = "don't consider ourselves criminals"
        $s11 = "What's wrong with my files?" fullword
        $s12 = "contact us and decrypt one"
        $s13 = "lockbitaptawjl6udhpd323uehekiyatj6ftcxmkwe5sezs4fqgpjpid.onion"
        $s14 = "binance.com/en/buy-Bitcoin"

        $btc = /[13][a-km-zA-HJ-NP-Z1-9]{25,34}/ fullword ascii wide
    condition:
        2 of them
}
'''

escanear(regla, 'ejercicio2')


---
# Ejercicio extra · Cazar una webshell  *(para hacer en casa)*

No da tiempo en clase. Lo tienes aquí para cuando quieras.

En `extra/` hay dos ficheros: una muestra con las cadenas características del panel **c99**,
y un `.php` legítimo que es tu **control de falsos positivos**.

> La muestra **no es una webshell funcional**: son solo las cadenas, para que puedas probar
> la regla sin manejar código malicioso de verdad.

### Nivel 1
Ejecuta. Debe detectar `muestra_c99.php` y **no** `inocente.php`.

### Nivel 2
Añade dos strings demasiado comunes y vuelve a ejecutar:

```
        $s9  = "system("
        $s10 = "echo"
```

Ahora cae también `inocente.php`: acabas de fabricar un falso positivo. Quítalas.

### Nivel 3
Añade el anclaje de formato: `$php = "<?php"` en `strings`, y `$php at 0 and` en la `condition`.


In [ ]:
regla = '''
rule mal_webshell_php_c99
{
    meta:
        author      = "tu nombre"
        description = "Webshell PHP de la familia c99"
    strings:
        $s0 = "<b>HEXDUMP:</b>"
        $s1 = "$filestealth"
        $s2 = "Server-status variables:"
    condition:
        filesize < 1MB and 2 of them
}
'''

escanear(regla, 'extra')


---
# Tu turno · escribe una regla desde cero

Crea tus propios ficheros y prueba a detectarlos. O sube los tuyos con el icono de la
carpeta de la izquierda y escanéalos.

**Las tres reglas de oro:**
1. Una string demasiado común = falso positivo.
2. Una string demasiado concreta = falso negativo.
3. Prueba siempre contra ficheros legítimos antes de desplegar.


In [ ]:
os.makedirs(os.path.join(BASE, 'mio'), exist_ok=True)
open(os.path.join(BASE, 'mio', 'sospechoso.txt'), 'w').write(
    'Escribe aqui el texto que quieres detectar')
open(os.path.join(BASE, 'mio', 'inocente.txt'), 'w').write(
    'Y aqui un fichero normal, que NO debe saltar')

regla = '''
rule mi_primera_regla
{
    strings:
        $a = "detectar"
    condition:
        $a
}
'''

escanear(regla, 'mio')


---
## Para seguir después de la clase

- **Reto:** coge un correo de phishing de tu carpeta de spam, súbelo aquí y escribe una regla
  que lo detecte sin marcar tus correos normales.
- Súbela a [yaraify.abuse.ch](https://yaraify.abuse.ch) y mira si genera falsos positivos.
- `#100DaysOfYARA` — una regla al día, reto de la comunidad.
- [yara.readthedocs.io](https://yara.readthedocs.io) — la documentación oficial, corta y legible.

El siguiente escalón: **el módulo `pe`**. Dejas de mirar el texto del fichero y empiezas a mirar
cómo está construido por dentro.
